# Experimental Feature Selection into XGBoost training

This work managed scores vs ideal strong case
| Metric | Now |
|---|---:|
| **ROC-AUC** | ~0.69–0.70 |
| **PR-AUC** | ~0.41–0.43 |
| **Precision (positive class)** | ~0.38 |
| **Recall (positive class)** | ~0.64 |
| **F1-score (positive class)** | ~0.48 |

Idea to test in the future to try to balance the dataset. The XGBoost is balancing the dataset by itself\
(scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1))\
But it could add some value

next step on what to focus more again:
- feature engineering
    - modulo day
    - post shift day
- feature selection / noise reduction
- threshold tuning
- hyperparameter tuning
- imbalance experiments if still needed

## Load data

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/processed/sample.csv", sep="|")
# df = pd.read_csv("../../data/processed/processed_joined_dataset.csv", sep="|")
df.head()

,lineID,day,pid,adFlag,availability,competitorPrice,click,basket,order,price,...,rrp,has_competitor,campaignIndex_A,campaignIndex_B,campaignIndex_C,price_diff_competitor,price_ratio_competitor,price_pct_diff_competitor,is_post_shift_day,price_diff_vs_previous_available_day
0,978899,39,9624,0,1,17.19,1,0,0,19.89,...,21.51,1,0,0,0,2.70,1.1571,15.71,1,0.0
1,1267035,47,3969,1,1,18.13,1,0,0,20.85,...,26.07,1,0,0,0,2.72,1.1500,15.00,1,0.0
2,297914,14,16633,0,1,15.06,0,0,1,16.45,...,23.98,1,0,1,0,1.39,1.0923,9.23,0,0.0
3,2554963,87,20147,0,1,4.36,1,0,0,5.17,...,5.45,1,0,0,0,0.81,1.1858,18.58,1,0.0
4,2739211,92,14326,0,1,NaN,0,0,1,6.22,...,6.55,0,0,0,0,NaN,NaN,NaN,1,0.0


In [2]:
print(df.shape)
print(df.columns.tolist())

(200000, 29)
['lineID', 'day', 'pid', 'adFlag', 'availability', 'competitorPrice', 'click', 'basket', 'order', 'price', 'revenue', 'manufacturer', 'group', 'content', 'unit', 'pharmForm', 'genericProduct', 'salesIndex', 'category', 'rrp', 'has_competitor', 'campaignIndex_A', 'campaignIndex_B', 'campaignIndex_C', 'price_diff_competitor', 'price_ratio_competitor', 'price_pct_diff_competitor', 'is_post_shift_day', 'price_diff_vs_previous_available_day']


In [3]:
df.isna().sum().sort_values(ascending=False)


competitorPrice                         7195
price_pct_diff_competitor               7195
price_diff_competitor                   7195
price_ratio_competitor                  7195
category                                6290
price_diff_vs_previous_available_day    2653
availability                               0
basket                                     0
click                                      0
day                                        0
pid                                        0
adFlag                                     0
lineID                                     0
group                                      0
manufacturer                               0
revenue                                    0
price                                      0
order                                      0
content                                    0
pharmForm                                  0
unit                                       0
has_competitor                             0
rrp       

## Handling missing Values

### Competitor Price
we keep them as is, because we have a flag feature has_competitor and that flags for us if the value is missing.
this should work for a decision tree type model

### Category
because it is a categorical variable we can replace NaN with new category unknown. Same Same but different xd

In [4]:
df["category"] = df["category"].fillna("unknown")

### Previous day price diff
this na values originate from the fact that there is no previous day price available (first day etc...) so we replace the missing values with neutral number 0. But this is only acceptable if we provide flag feature prev_day_price_missing. Once again works for decision tree style models.

In [5]:
df["prev_day_price_missing"] = df["price_diff_vs_previous_available_day"].isna().astype(int)
df["price_diff_vs_previous_available_day"] = df["price_diff_vs_previous_available_day"].fillna(0)

## Adding modulo Day feature

I noticed weekly repetition in customer behaviour and therefore i will add new
feature that will be modulo 7 of the day number to give
the model an idea about the week.

In [6]:
df["modulo_day"] = df.day % 7

# df.sort_values('day').loc[:,['day', 'modulo_day']]  # only for check

## Target definition and removing obvious leakage

In [7]:
target = "order"

drop_cols = [
    "order",      # target
    "click",      # leakage
    "basket",     # leakage
    "revenue",    # leakage
    "lineID",     # identifier
    "pid",        # identifier / memorization risk
    "unit",        # useless (for now)
    "rrp",              # keep only as ratio input
    "content",          # replace by content_parsed
]

## Perform Feature Selection

- remove obvious bad columns manually
- separate numeric and categorical
- mutual_info_classif for a first relevance ranking

In [8]:
print("our features: ", df.columns)
print("target: ", target)
print("Already determined drops: ", drop_cols)

our features:  Index(['lineID', 'day', 'pid', 'adFlag', 'availability', 'competitorPrice',
       'click', 'basket', 'order', 'price', 'revenue', 'manufacturer', 'group',
       'content', 'unit', 'pharmForm', 'genericProduct', 'salesIndex',
       'category', 'rrp', 'has_competitor', 'campaignIndex_A',
       'campaignIndex_B', 'campaignIndex_C', 'price_diff_competitor',
       'price_ratio_competitor', 'price_pct_diff_competitor',
       'is_post_shift_day', 'price_diff_vs_previous_available_day',
       'prev_day_price_missing', 'modulo_day'],
      dtype='str')
target:  order
Already determined drops:  ['order', 'click', 'basket', 'revenue', 'lineID', 'pid', 'unit', 'rrp', 'content']


### Zero-Variance Feature Removal
I will do this first during feature selection because zero-variance features have
the same value in all observations, so they carry no useful information for predicting the target.

In [9]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif, SelectKBest

drop_cols +=  [
    "day",                    # replaced by is_post_shift_day + modulo_day
    "competitorPrice",        # redundant if using competitor-derived features
    "price_diff_competitor",  # redundant with ratio / pct diff
    "price_pct_diff_competitor",
]

X = df.drop(columns=drop_cols)
y = df[target]

categorical_cols = ["manufacturer", "group", "pharmForm", "category"]
for col in categorical_cols:
    X[col] = X[col].astype("category")

# only numeric columns
X_num = X.select_dtypes(include=[np.number]).copy()

vt = VarianceThreshold(threshold=0.0)
vt.fit(X_num)

dropped_features = X_num.columns[~vt.get_support()]
kept_features = X_num.columns[vt.get_support()]

print("Dropped zero-variance features:", list(dropped_features))
print("Kept numeric features:", list(kept_features))

X = X.drop(columns=list(dropped_features))

# we did nto dropped anything meaning all zero variance features were dropped
# by our manual assessment while creating drop_cols list that was derived from observations during EDA

Dropped zero-variance features: []
Kept numeric features: ['adFlag', 'availability', 'price', 'genericProduct', 'salesIndex', 'has_competitor', 'campaignIndex_A', 'campaignIndex_B', 'campaignIndex_C', 'price_ratio_competitor', 'is_post_shift_day', 'price_diff_vs_previous_available_day', 'prev_day_price_missing', 'modulo_day']


### Mutual information feature evaluation

I will do this next because the zero-variance check only tells me whether a feature changes at all,
but not whether it is actually useful for predicting the target.
Mutual information helps measure how much information each feature gives about the target variable,
so it is a good next step to identify which features seem more relevant and which ones may be weak.

In [10]:
# temporary copy only for mutual information evaluation
X_mi = X.copy()

# fill categorical/string missing values
for col in X_mi.select_dtypes(include=["object", "category", "string"]).columns:
    X_mi[col] = X_mi[col].fillna("unknown").astype("category").cat.codes

# fill numeric missing values temporarily for MI
for col in X_mi.select_dtypes(include=[np.number]).columns:
    X_mi[col] = X_mi[col].fillna(-999)

# compute mutual information
mi_scores = mutual_info_classif(
    X_mi,
    y,
    discrete_features="auto",
    random_state=42
)

mi_df = pd.DataFrame({
    "feature": X_mi.columns,
    "mutual_info": mi_scores
}).sort_values("mutual_info", ascending=False)

print(mi_df.to_string(index=False))

                             feature  mutual_info
                   is_post_shift_day     0.035315
              price_ratio_competitor     0.029486
                      has_competitor     0.029293
                        availability     0.027953
                               group     0.026002
                               price     0.021184
                            category     0.020412
                        manufacturer     0.019489
                          salesIndex     0.016009
                           pharmForm     0.013842
                              adFlag     0.011724
                          modulo_day     0.006806
                      genericProduct     0.004946
                     campaignIndex_A     0.001622
price_diff_vs_previous_available_day     0.001191
                     campaignIndex_B     0.000797
                     campaignIndex_C     0.000000
              prev_day_price_missing     0.000000


Mutual information evaluation showed that the strongest features were is_post_shift_day, price_ratio_competitor, has_competitor, availability, group, and price, indicating that time shift, competitor positioning, stock status, and product grouping carry the most information about the target. On the other hand, features such as campaignIndex_C and prev_day_price_missing showed zero mutual information, while campaignIndex_B and price_diff_vs_previous_available_day showed only very weak signal. These features are therefore good candidates for removal in the next feature selection iteration.

In [11]:
extra_drop_cols = [
    "campaignIndex_C",
    "prev_day_price_missing",
]
X = X.drop(columns=extra_drop_cols)

These features are strongly correlated and might be redundant to use them all

## Split data
Before training the model, the dataset is split into training and test sets using an 80/20 ratio. A stratified split is applied so that the proportion of the target class remains almost the same in both subsets. This is important because the target is moderately imbalanced, and keeping the class distribution consistent ensures that model training and later evaluation are more reliable.

In [12]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(160000, 16) (40000, 16)
0.25585625 0.25585


### check if we have missing values
if 0 missing values we can proceed otherwise we need to fix it

In [13]:
print("Missing values in X_train before imputation:")
print(X_train.isna().sum().sort_values(ascending=False))

print("\nMissing values in X_test before imputation:")
print(X_test.isna().sum().sort_values(ascending=False))

Missing values in X_train before imputation:
price_ratio_competitor                  5804
adFlag                                     0
price                                      0
availability                               0
group                                      0
pharmForm                                  0
genericProduct                             0
manufacturer                               0
salesIndex                                 0
category                                   0
campaignIndex_A                            0
has_competitor                             0
campaignIndex_B                            0
is_post_shift_day                          0
price_diff_vs_previous_available_day       0
modulo_day                                 0
dtype: int64

Missing values in X_test before imputation:
price_ratio_competitor                  1391
adFlag                                     0
price                                      0
availability                              

## Training XGBoost
An XGBoost classifier was trained on the selected numeric features to predict whether a product would be purchased (order = 1). The model was configured with a set of initial hyperparameters and adjusted for class imbalance using scale_pos_weight, so that the minority class received more attention during training. After fitting the model on the training data, predictions and predicted probabilities were generated for the test set, and the model performance was evaluated using ROC-AUC, PR-AUC, the classification report, and the confusion matrix.

In [14]:
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report, confusion_matrix

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss",
    scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
    enable_categorical=True
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))
print("PR-AUC:", average_precision_score(y_test, y_pred_proba))
print("\nClassification report:")
print(classification_report(y_test, y_pred))
print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

ROC-AUC: 0.6950212701347889
PR-AUC: 0.4106122828478997

Classification report:
              precision    recall  f1-score   support

           0       0.85      0.61      0.71     29766
           1       0.37      0.68      0.48     10234

    accuracy                           0.63     40000
   macro avg       0.61      0.64      0.60     40000
weighted avg       0.73      0.63      0.65     40000


Confusion matrix:
[[18172 11594]
 [ 3288  6946]]


## Interpretation of XGBoost Results
The XGBoost model achieved a ROC-AUC of 0.695 and a PR-AUC of 0.411, which means its clearly better than random guessing.

For the positive class (order = 1), the model reached a recall of 0.68, so it detects a relatively large share of actual purchases. However, its precision is only 0.37, which means many predicted purchases are false positives. The F1-score of 0.48 reflects this trade-off.

The confusion matrix shows the same pattern: the model finds many real purchases, but also predicts too many purchases incorrectly. Overall, the model is a usable baseline, but the main weakness is low precision.

Ideas for Improvement
test log(price) instead of raw price
remove weakest features (campaignIndex_C, prev_day_price_missing, possibly others)
inspect XGBoost feature importance
improve grouping of manufacturer, group, and category
refine competitor-related features
tune XGBoost hyperparameters
test classification threshold instead of using only 0.5